# Week 09: vLLM Serving & Benchmarks

# Requirements: pip install vllm openai requests numpy pandas

# ⚠️ REQUIRES: GPU for vLLM; fallback = Ollama

This notebook starts a vLLM **OpenAI-compatible** server as a subprocess, benchmarks
latency and throughput at batch sizes 1 to 8, and compares the same model served by
**Ollama**: producing the speed half of your quality-vs-cost report. If you have no
CUDA GPU, the vLLM cells skip cleanly and the Ollama path still yields real numbers.


## 0. Setup: repo root on the path + seeded RNG


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import os, time, json, subprocess, urllib.request
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

try:
    from openai import OpenAI
    _OPENAI_OK = True
except Exception as _e:  # pragma: no cover
    OpenAI = None
    _OPENAI_OK = False
    print("openai SDK not installed:", _e)

print("imports ok, openai SDK:", _OPENAI_OK)


## 1. Do we have a GPU for vLLM?

vLLM is a CUDA serving engine. Without CUDA we fall back to Ollama (which runs on CPU,
Metal, or CUDA via llama.cpp).


In [ ]:
def cuda_available():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

HAS_CUDA = cuda_available()
print("CUDA available:", HAS_CUDA)


## 2. Start the vLLM server (subprocess)

`vllm.entrypoints.openai.api_server` exposes an OpenAI-compatible `/v1` endpoint.
`--gpu-memory-utilization 0.85` tells PagedAttention how much VRAM it may use; capping
`--max-model-len` saves KV-cache memory. We launch it detached so the notebook keeps
running, and remember the pid so we can stop it at the end.


In [ ]:
VLLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
VLLM_PORT = 8000
VLLM_URL = f"http://localhost:{VLLM_PORT}/v1"

vllm_proc = None
def start_vllm():
    global vllm_proc
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", VLLM_MODEL,
        "--port", str(VLLM_PORT),
        "--gpu-memory-utilization", "0.85",
        "--max-model-len", "2048",
    ]
    vllm_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return vllm_proc

if HAS_CUDA:
    start_vllm()
    print("vLLM server launching on port", VLLM_PORT, "(first load takes a minute)...")
else:
    print("No CUDA GPU, skipping vLLM; Ollama fallback below will run instead.")


## 3. Wait for readiness

Poll the server's `/models` endpoint until it answers, with a timeout.


In [ ]:
def wait_ready(url, timeout=300):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url + "/models", timeout=2) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False

VLLM_READY = False
if vllm_proc is not None:
    VLLM_READY = wait_ready(VLLM_URL)
    print("vLLM ready:", VLLM_READY)
else:
    print("vLLM not started (no GPU).")


## 4. The benchmark harness

We measure two numbers per batch size: **mean latency** (ms to first completion, here
measured as full round-trip) and **throughput** (aggregate completion tokens per second
across the concurrent batch). A thread pool issues `batch_size` requests at once, this
is where continuous batching shows up: the GPU keeps working while others wait.


In [ ]:
PROMPTS = [
    "Classify this support ticket into one of: tracking, damage, refund, documents, customs, billing. Ticket: Customs is holding shipment S0001234 at Long Beach. What documents do you need?",
    "Summarize this shipment note in one sentence: Shipment S0009876 of electronics (850 kg) from Chicago to Memphis is in transit and on time.",
    "What is the category of this ticket? Ticket: The pallet arrived damaged, the boxes are crushed.",
]

client = None
if _OPENAI_OK:
    client = OpenAI(base_url=VLLM_URL, api_key="EMPTY")

def latency_ms(client, model_name, prompt, max_tokens=32):
    t0 = time.time()
    resp = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.0,
    )
    dt = (time.time() - t0) * 1000
    ntok = resp.usage.completion_tokens
    return dt, ntok

def benchmark(client, model_name, batch_size, rounds=2):
    from concurrent.futures import ThreadPoolExecutor
    latencies, total_tok, wall_total = [], 0, 0.0
    for _ in range(rounds):
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=batch_size) as ex:
            futures = [ex.submit(latency_ms, client, model_name, PROMPTS[i % len(PROMPTS)])
                       for i in range(batch_size)]
            for f in futures:
                dt, ntok = f.result()
                latencies.append(dt)
                total_tok += ntok
        wall_total += (time.time() - t0)
    mean_lat = float(np.mean(latencies))
    throughput = total_tok / wall_total if wall_total > 0 else 0.0
    return mean_lat, throughput


## 5. vLLM sweep at batch sizes 1 / 2 / 4 / 8

Watch latency rise and throughput rise faster, the shape of the tradeoff is the
deliverable.


In [ ]:
vllm_results = {}
if VLLM_READY and client is not None:
    for bs in [1, 2, 4, 8]:
        lat, thr = benchmark(client, VLLM_MODEL, bs)
        vllm_results[bs] = {"latency_ms": round(lat, 1), "tokens_per_sec": round(thr, 1)}
        print(f"vLLM batch={bs}: latency {lat:.1f} ms, throughput {thr:.1f} tok/s")
else:
    print("Skipping vLLM benchmark, server not ready.")


## 6. Ollama fallback (same model, same protocol)

Ollama serves `qwen2.5:1.5b` behind an OpenAI-compatible `/v1` endpoint at port 11434.
The client is the *same* OpenAI SDK, only the base URL and model name change, which is
exactly why OpenAI compatibility is a superpower.


In [ ]:
OLLAMA_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "qwen2.5:1.5b"

def ollama_ready():
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

OLLAMA_RUNNING = ollama_ready()
print("Ollama running:", OLLAMA_RUNNING)
if not OLLAMA_RUNNING:
    print("Tip: start Ollama with `ollama serve`, then `ollama pull qwen2.5:1.5b`.")


In [ ]:
ollama_client = None
if _OPENAI_OK and OLLAMA_RUNNING:
    ollama_client = OpenAI(base_url=OLLAMA_URL, api_key="EMPTY")

ollama_results = {}
if ollama_client is not None:
    for bs in [1, 2, 4]:
        lat, thr = benchmark(ollama_client, OLLAMA_MODEL, bs)
        ollama_results[bs] = {"latency_ms": round(lat, 1), "tokens_per_sec": round(thr, 1)}
        print(f"Ollama batch={bs}: latency {lat:.1f} ms, throughput {thr:.1f} tok/s")
else:
    print("Skipping Ollama benchmark, Ollama not reachable.")


## 7. The speed table

One row per (engine, batch size). This is the raw material for the report.


In [ ]:
def results_frame(res, engine):
    if not res:
        return pd.DataFrame()
    return pd.DataFrame([
        {"engine": engine, "batch_size": bs,
         "latency_ms": v["latency_ms"], "tokens_per_sec": v["tokens_per_sec"]}
        for bs, v in res.items()
    ])

frames = [results_frame(vllm_results, "vLLM"), results_frame(ollama_results, "Ollama")]
frames = [f for f in frames if not f.empty]
bench = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
    columns=["engine", "batch_size", "latency_ms", "tokens_per_sec"])
print(bench.to_string(index=False))


## 8. Quality-vs-cost

Cost = tokens x price. Throughput is the lever that decides cost at fixed quality: an
engine that serves the *same* tokens per second with less hardware, or more tokens per
second on the *same* hardware, is strictly cheaper per task. We fold in the Week 9
quantization facts (4-bit ≈ 0.5 bytes/param) to tie speed to memory.


In [ ]:
ASSUMED_PRICE_PER_1K = 0.002 # USD per 1k tokens, illustrative, use your provider's number

def best_of(res):
    return max(res.values(), key=lambda v: v["tokens_per_sec"]) if res else None

vllm_best = best_of(vllm_results)
oll_best = best_of(ollama_results)

speedup = None
if vllm_best and oll_best and oll_best["tokens_per_sec"] > 0:
    speedup = vllm_best["tokens_per_sec"] / oll_best["tokens_per_sec"]

cost_per_1m = None
if vllm_best:
    cost_per_1m = ASSUMED_PRICE_PER_1K * 1000 # 1M tokens = 1000 x 1k

print("best vLLM throughput (tok/s):", vllm_best)
print("best Ollama throughput (tok/s):", oll_best)
print("vLLM / Ollama speedup: ", None if speedup is None else round(speedup, 2))
print("illustrative USD per 1M tokens:", cost_per_1m)
print("4-bit NF4 VRAM (Week 9 fact): ", round(1.54e9 * 0.5 / 1e9, 2), "GB")


## 9. Takeaway

vLLM's PagedAttention + continuous batching exists for exactly this curve: under
concurrency it sustains throughput a naive batcher cannot. Ollama is the simplest path to
a local model; vLLM is the throughput workhorse. The report you write Friday picks the
engine and the quantization *together*, the point where the cost curve and the quality
curve (from notebook 01) cross.


In [ ]:
# FINAL number: best measured throughput (tokens/sec). 0.0 means nothing ran here, 
# still a number, and it tells you to go run Ollama (or add a GPU) before Friday.
if vllm_best is not None:
    final_thr = vllm_best["tokens_per_sec"]
elif oll_best is not None:
    final_thr = oll_best["tokens_per_sec"]
else:
    final_thr = 0.0

# stop the server now that measurement is done
if vllm_proc is not None:
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=10)
    except Exception:
        vllm_proc.kill()
    print("vLLM server stopped.")

print(f"BEST_THROUGHPUT_TOK_PER_SEC={final_thr:.2f}")
